Every time you talk to ChatGPT / Grok / Claude / Llama, it forgets everything after you send the message.
Each question is completely new — it does not remember what you said 2 messages ago.We give the LLM a notebook where we write down what happened before.
Every time we ask a new question, we also show the notebook so the LLM can remember the previous conversation.
That's basically 100% of what "memory" means in LangChain.

Example A – Very common situation
User: "My name is Vikash"
AI: "Nice to meet you Vikash"
User (next message): "What is my name?"
→ Without memory → AI says "I don't know"
→ With memory → AI says "Your name is Vikash"

In [19]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.chat_models import init_chat_model

llm = init_chat_model("llama3.2", model_provider="ollama")
# 1. Create Prompt Template
prompt = ChatPromptTemplate.from_template(
    """
Question: {question}
"""
)
# 3. Create Output Parser
output_parser = StrOutputParser()

# 4. Build the Chain using LCEL (pipe | operator)
chain = prompt | llm | output_parser


response = chain.invoke({
    "question": "Hi my name is vikash?"
})
print(response)

response = chain.invoke({
    "question": "what is my name?"
})
print(response)

Hi Vikash! It's nice to meet you. Is there something I can help you with or would you like to chat?
I don't know your name. I'm a large language model, I don't have any information about you or your personal identity. I'm here to help answer your questions and provide information, but I need more context from you to be able to assist you further. Is there anything else I can help with?


Conversation memory in LangChain (as of March 2026) refers to the mechanism that lets a chain, agent, or chatbot remember previous messages in a conversation.The recommended and dominant approach in current LangChain (v0.3+ era, still true in 2026) is RunnableWithMessageHistory.It replaced most of the older ConversationChain, ConversationBufferMemory, ConversationSummaryMemory, etc. (those are either deprecated or only used in very specific legacy cases).it Works with any runnable (simple LLM calls, RAG chains, agents, custom logic, streaming, async…).


session_id is is basically the identifier (key) that tells LangChain which conversation history to load / save for a particular chat session.

MessagesPlaceholder("history") in LangChain automatically inserts all the previous messages (the conversation history) when using RunnableWithMessageHistory.

RunnableWithMessageHistory does:
Loads the history for session Id ,
Takes all past messages (list of BaseMessage objects: HumanMessage, AIMessage, etc.),
Puts them into the prompt at the position ofMessagesPlaceholder("history"),
Adds the current message→ becomes a HumanMessage,
Sends the full list of messages to the model

After the model replies → it appends both the new human message + new AI message to the history store (so next time they're included again)

config parameter you pass to .invoke(), .stream(), .batch() etc. on a RunnableWithMessageHistory is a dictionary that controls runtime behavior — most importantly, which conversation history to load/save.config is always the second positional argument (or keyword config=) in .invoke(input, config=...).The key part is  "configurable" → that's how LangChain's configurable runnables work.By default, RunnableWithMessageHistory looks for session_id inside config["configurable"].


ChatMessageHistory is the simplest, most common in-memory implementation of chat message history in LangChain.
It's a concrete class that stores a list of messages (HumanMessage, AIMessage, SystemMessage, etc.) entirely in RAM — no database, no persistence after the program restarts.Popular persistent alternatives (all inherit from BaseChatMessageHistory):

RedisChatMessageHistory

PostgresChatMessageHistory

DynamoDBChatMessageHistory

MongoDBChatMessageHistory

UpstashRedisChatMessageHistory

SQLChatMessageHistory

corelangchain-core package is Base package to provide  Message types, PromptTemplate, OutputParser, callbacks, etc.— the foundation .


langchain_community is the official Python package  that contains community-maintained / third-party integrations for the LangChain ecosystem like  document loaders, vector stores, chat message histories, embeddings, LLMs/chat models from external providers, tools, etc.

install it by 'uv add langchain_community'

In [20]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_ollama import OllamaLLM   # Note: this is the non-chat version

# 1. Create the prompt (with memory placeholder)
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly and helpful assistant."),
    MessagesPlaceholder(variable_name="chat_history"),   # ← memory goes here
    ("human", "{question}"),
])

# 2. Choose model
model = OllamaLLM(model="llama3.2")

# 3. Combine prompt + model
chain = prompt | model

# 4. Very simple memory storage (just a dictionary)
store = {}   # key = session_id, value = chat history object

def get_memory(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# 5. Add memory to the chain
chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_memory,
    input_messages_key="question",       # matches prompt variable
    history_messages_key="chat_history", # matches MessagesPlaceholder
)

# ────────────────────────────────────────────────
#           Now let's talk (like a real chat)
# ────────────────────────────────────────────────

# First message
response1 = chain_with_memory.invoke(
    {"question": "Hi! My name is vikash."},
    config={"configurable": {"session_id": "conversation-abc123"}}
)

print("AI:", response1)

# Second message – should remember the name
response2 = chain_with_memory.invoke(
    {"question": "what is my name?"},
    config={"configurable": {"session_id": "conversation-abc123"}}
)

print("AI:", response2)

# Third message
response3 = chain_with_memory.invoke(
    {"question": "What did I say in my first message?"},
    config={"configurable": {"session_id": "conversation-abc123"}}
)

print("AI:", response3)

AI: Hi Vikash! It's nice to meet you. I'm here to help with any questions or problems you may have. How can I assist you today?
AI: Nice try, Vikash! However, as a friendly assistant, I need to clarify that I don't actually know your real name. You just told me it's "Vikash", but I'm not storing any personal information about you. I'm here to provide helpful and anonymous assistance. Would you like to start fresh and ask for help with something specific or explore a topic?
AI: You mentioned that the system didn't know your name, but then later asked what you said in your first message. You initially told me "what is my name?" and I should have responded by clarifying that I was the one being addressed, rather than asking for confirmation of a question.

So, to correct myself: You initially said "Hi! My name is vikash" in your first message. Does that sound right?


Tool Calling (also known as Function Calling) is the key feature that allows AI to take actions in the real world instead. Action like Read Excel / CSV files, Check order status in company system
Process refunds or returns,
Generate invoices,
Query CRM (Customer Relationship Management)

Without Tool Calling:
User: "What's the current stock price of Tesla?"
AI: "As of my last training, it was around $250..." (outdated or wrong)
LLMs have old knowledge (cutoff date). Tool calling gives them real-time access (stock prices)
With Tool Calling:
User: Same question
AI: Calls get_stock_price("TSLA") tool → Gets live price → Gives accurate answer + analysis

In [9]:
from langchain_core.tools import tool
from langchain_ollama import ChatOllama

# Two Tools
@tool
def multiply(a: int, b: int):
    """Multiply two numbers"""
    return a * b

@tool
def add(a: int, b: int):
    """Add two numbers"""
    return a + b

# Setup
llm = ChatOllama(model="llama3.2", temperature=0)
llm_with_tools = llm.bind_tools([multiply, add])

# Question
response = llm_with_tools.invoke("What is 5 multiplied by 3? Then add 10 to it.")

# Execute tool and print result
if response.tool_calls:
    for call in response.tool_calls:
        tool_name = call["name"]
        args = call["args"]
        
        if tool_name == "multiply":
            result = multiply.invoke(args)
        elif tool_name == "add":
            result = add.invoke(args)
            
        print(f"Tool Result: {result}")
else:
    print("No tool called")

Tool Result: 15
Tool Result: 25


Temperature is a parameter that controls the creativity vs accuracy of the LLM’s responses.Temperature = 0 → Student who always gives exact textbook answer
Temperature = 1 → Student who gives creative, sometimes wrong answers.

Tool calling needs exact JSON format
Even small creativity can break the tool call so we used temprature=0,

Step-by-step: LLM → decides tool → gives perfect arguments



Temprature (0) No creativityVery focused, deterministic Tool calling, Math, Code

temp (0.5 - 0.7) BalancedNormal, natural and General chatting

temp (1.0)High creativityCreative, diverse, riskyStory writing, Brainstorming

How does the LLM know when to call a tool?

The LLM does not know automatically.
We teach it by giving it tool descriptions and a special format.

You define tools with clear description. 

In [10]:
@tool
def multiply(a: int, b: int):
    """Multiply two numbers"""   # ← This description is very important
    return a * b

When you send a question, the LLM receives:

The question
All tool descriptions (name + what it does + arguments)

LLM decides based on:
The description you wrote
The question
Its training on when to use tools

Question: "What is 15 multiplied by 4?"

What LLM sees internally:

Tool available: multiply(a, b) → "Multiply two numbers"
Question is about multiplication

→ LLM thinks: "This matches the multiply tool"
→ So it calls the tool